In [ ]:
!python --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#import pandas to read the csv file
import pandas as pd
import numpy as np

In [ ]:
#Importing Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
# from sklearn.externals import joblib

In [ ]:
import joblib

In [ ]:
#reading the y.txt file which contains labels
y = pd.read_csv('/content/drive/MyDrive/y.txt', sep = " ", header=None, names=["Complexity", "Node Number", "Language"])

In [ ]:
y

In [ ]:
x = np.load('/content/drive/MyDrive/x.npy')

In [ ]:
x

In [ ]:
df = []
for i in range(len(x)):
  emb = x[i].tolist()
  # emb.append(y[i])
  df.append(emb)
col = []
for i in range(128):
  col.append('emb'+str(i+1))
# col.append('classes')
df = pd.DataFrame(df, columns=col)

In [ ]:
df

In [ ]:
df_y = pd.read_csv('/content/drive/MyDrive/y.txt', sep = " ", header=None, names=["Complexity", "Node Number", "Language"])

In [ ]:
df['Complexity'] = df_y['Complexity']

In [ ]:
import seaborn as sns
sns.countplot(df['Complexity'])

# **Resampling - Oversampling**

In [ ]:
from sklearn.utils import resample
#create two different dataframe of majority and minority class 
df_majority = df[(df['Complexity'] == 'O(N)')] 
df_minority_1 = df[(df['Complexity'] == 'O(N2)')] 
df_minority_2 = df[(df['Complexity'] == 'O(NlogN)')] 
df_minority_3 = df[(df['Complexity'] == 'O(logN)')] 
df_minority_4 = df[(df['Complexity'] == 'O(N3)')]
df_minority_5 = df[(df['Complexity'] == 'O(Nd)')]
df_minority_6 = df[(df['Complexity'] == 'O(2n)')]
df_minority_7 = df[(df['Complexity'] == 'O(1)')]
df_minority_8 = df[(df['Complexity'] == 'O(N!)')]
df_minority_9 = df[(df['Complexity'] == 'O(sqrt(N))')]

# upsample minority class
minority = [df_minority_1, df_minority_2, df_minority_3, df_minority_4, df_minority_5, df_minority_6, df_minority_7, df_minority_8, df_minority_9]
temp = []
for i in minority:
  df_minority_upsampled = resample(i, 
                                  replace=True,    # sample with replacement
                                  n_samples= 375, # to match majority class
                                  random_state=42)  # reproducible results
  temp.append(df_minority_upsampled)
# Combine majority class with upsampled minority class
df_upsampled = df_majority
for i in temp:
  df_upsampled = pd.concat([i, df_upsampled])

In [ ]:
df_upsampled['Complexity'].value_counts()

In [ ]:
sns.countplot(df_upsampled['Complexity'])

In [ ]:
df = df_upsampled
# df = oversampled
factor = pd.factorize(df['Complexity'])
df.Complexity= factor[0]
definitions = factor[1]
print(df.Complexity.head())
print(definitions)

In [ ]:
#Splitting the data into independent and dependent variables
X = df.iloc[:,0:128].values
y = df.iloc[:,128].values
print('The independent features set: ')
print(X[:5,:])
print('The dependent variable: ')
print(y[:5])

In [ ]:
y = df['Complexity']

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
# Creating the Training and Test set from data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.35, random_state = 21)

In [ ]:
# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
model_results = pd.DataFrame()
model_results['models'] = ['Random Forest','Adaboost','XGBoost','KNN','Logistic Regression','Naive Bayes']
acc = []
prec = []
rec = []
f1 = []
kap = []
roc = []

# Random forest

In [ ]:
# Fitting Random Forest Classification to the Training set
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 42)
classifier.fit(X_train, y_train)

In [ ]:
num_classes = len(set(df_upsampled['Complexity']))
num_classes

In [ ]:
# Predicting the Test set results
y_pred = classifier.predict(X_test)

In [ ]:
confusion_matrix(y_test, y_pred)

In [ ]:
# Get and reshape confusion matrix data
import matplotlib.pyplot as plt
matrix = confusion_matrix(y_test, y_pred)
matrix = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis]

# Build the plot
plt.figure(figsize=(16,7))
sns.set(font_scale=1.4)
sns.heatmap(matrix, annot=True, annot_kws={'size':10},
            cmap=plt.cm.Greens, linewidths=0.2)

# Add labels to the plot
class_names = ['O(1)',  'O(2n)',  'O(N!)',  'O(N)',  'O(N2)',  'O(N3)',  'O(Nd)',  'O(NlogN)', 'O(logN)','O(sqrt(N))']
tick_marks = np.arange(len(class_names))
tick_marks2 = tick_marks + 0.5
plt.xticks(tick_marks, class_names, rotation=25)
plt.yticks(tick_marks2, class_names, rotation=0)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix for Random Forest Model')
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

accuracy_score(y_test, y_pred)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
# from imblearn.under_sampling import RandomUnderSampler
# from imblearn.over_sampling import SMOTE
# smote_obj = SMOTE()
# X_train, y_train = smote_obj.fit_resample(X_train, y_train)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

In [ ]:
classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 42)
classifier.fit(X_train, y_train)

In [ ]:
# Predicting the Test set results
y_pred = classifier.predict(X_test)

In [ ]:
y_test

In [ ]:
y_pred

In [ ]:
accuracy = accuracy_score(y_test, y_pred) 
print("Accuracy: %.2f%%" % (accuracy * 100.0))

In [ ]:
predictions = [round(value) for value in y_pred]

In [ ]:
from sklearn.metrics import classification_report,precision_score,recall_score,f1_score,cohen_kappa_score
from sklearn import metrics
labels=['0','1','2','3','4']
print(classification_report(y_test, y_pred))
print(precision_score(y_test, y_pred,average='weighted'))
print(recall_score(y_test, y_pred,average='weighted'))
print(f1_score(y_test, y_pred,average='weighted'))
print(cohen_kappa_score(y_test, predictions))
#print("Kappa score:" + cohen_kappa_score(y_test, y_pred,average='weighted'))

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = classifier.predict_proba(X_test)
weighted_roc_auc_ovo = roc_auc_score(
    y_test, y_prob, multi_class="ovr", average="weighted"
)
weighted_roc_auc_ovo

In [ ]:
acc.append(accuracy_score(y_test, y_pred))
prec.append(precision_score(y_test, y_pred,average='weighted'))
rec.append(recall_score(y_test, y_pred,average='weighted'))
f1.append(f1_score(y_test, y_pred,average='weighted'))
kap.append(cohen_kappa_score(y_test, predictions))
roc.append(weighted_roc_auc_ovo)

In [ ]:
# joblib.dump(classifier, "/content/drive/MyDrive/Capstone/random_forest.joblib")

In [ ]:
# loaded_rf = joblib.load("/content/drive/MyDrive/Capstone/random_forest_GridSearch.joblib")

In [ ]:
# x_new = np.load('/content/drive/MyDrive/x_new.npy')
# df = []
# x = x_new
# for i in range(len(x)):
#     emb = x[i].tolist()
#     # emb.append(y[i])
#     df.append(emb)
# col = []
# for i in range(128):
#     col.append('emb'+str(i+1))
# # col.append('classes')
# df = pd.DataFrame(df, columns=col)

# X_new = df.iloc[:,0:128].values

In [ ]:
# y_pred = loaded_rf.predict(X_new)
# y_pred

# Adaboost

In [ ]:
from sklearn.datasets import make_gaussian_quantiles
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

In [ ]:
bdt_real = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=2), n_estimators=300, learning_rate=1
)

bdt_discrete = AdaBoostClassifier(
    DecisionTreeClassifier(max_depth=2),
    n_estimators=300,
    learning_rate=1.5,
    algorithm="SAMME",
)

In [ ]:

bdt_real.fit(X_train, y_train)
bdt_discrete.fit(X_train, y_train)


In [ ]:
y_pred = bdt_real.predict(X_test) 
y_pred1 = bdt_discrete.predict(X_test) 

In [ ]:
predictions = [round(value) for value in y_pred]

In [ ]:
print(classification_report(y_test, y_pred1))
print(accuracy_score(y_test, y_pred1))
print(precision_score(y_test, y_pred1,average='weighted'))
print(recall_score(y_test, y_pred1,average='weighted'))
print(f1_score(y_test, y_pred1,average='weighted'))
print(cohen_kappa_score(y_test, predictions))

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = bdt_real.predict_proba(X_test)
weighted_roc_auc_ovo = roc_auc_score(
    y_test, y_prob, multi_class="ovr", average="weighted"
)
weighted_roc_auc_ovo

In [ ]:
acc.append(accuracy_score(y_test, y_pred))
prec.append(precision_score(y_test, y_pred,average='weighted'))
rec.append(recall_score(y_test, y_pred,average='weighted'))
f1.append(f1_score(y_test, y_pred,average='weighted'))
kap.append(cohen_kappa_score(y_test, predictions))
roc.append(weighted_roc_auc_ovo)

# XGBoost

In [ ]:
import xgboost as xgb

from sklearn import datasets

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder 

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score

model = XGBClassifier() 
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test) 

predictions = [round(value) for value in y_pred]

In [ ]:
accuracy = accuracy_score(y_test, predictions) 

print("Accuracy: %.2f%%" % (accuracy * 100.0))

In [ ]:
from sklearn.metrics import classification_report,precision_score,recall_score,f1_score,cohen_kappa_score
from sklearn import metrics
labels=['0','1','2','3','4']
print(classification_report(y_test, y_pred))
print(precision_score(y_test, predictions,average='weighted'))
print(recall_score(y_test, predictions,average='weighted'))
print(f1_score(y_test, predictions,average='weighted'))
print(cohen_kappa_score(y_test, predictions))

In [ ]:
# Get and reshape confusion matrix data
import matplotlib.pyplot as plt
matrix = confusion_matrix(y_test, y_pred)
matrix = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis]

# Build the plot
plt.figure(figsize=(16,7))
sns.set(font_scale=1.4)
sns.heatmap(matrix, annot=True, annot_kws={'size':10},
            cmap=plt.cm.Greens, linewidths=0.2)

# Add labels to the plot
class_names = ['O(1)',  'O(2n)',  'O(N!)',  'O(N)',  'O(N2)',  'O(N3)',  'O(Nd)',  'O(NlogN)', 'O(logN)','O(sqrt(N))']
tick_marks = np.arange(len(class_names))
tick_marks2 = tick_marks + 0.5
plt.xticks(tick_marks, class_names, rotation=25)
plt.yticks(tick_marks2, class_names, rotation=0)
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.title('Confusion Matrix for XG Boost Model')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test)
weighted_roc_auc_ovo = roc_auc_score(
    y_test, y_prob, multi_class="ovr", average="weighted"
)
weighted_roc_auc_ovo

In [ ]:
acc.append(accuracy_score(y_test, y_pred))
prec.append(precision_score(y_test, y_pred,average='weighted'))
rec.append(recall_score(y_test, y_pred,average='weighted'))
f1.append(f1_score(y_test, y_pred,average='weighted'))
kap.append(cohen_kappa_score(y_test, predictions))
roc.append(weighted_roc_auc_ovo)

# KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
knn_clf=KNeighborsClassifier()
knn_clf.fit(X_train,y_train)
ypred=knn_clf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.metrics import classification_report
result1 = classification_report(y_test, ypred)
print("Classification Report:",)
print (result1)
result2 = accuracy_score(y_test,ypred)
print("Accuracy:",result2)
print(precision_score(y_test, ypred,average='weighted'))
print(recall_score(y_test, ypred,average='weighted'))
print(f1_score(y_test, ypred,average='weighted'))
print(cohen_kappa_score(y_test, ypred))

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = knn_clf.predict_proba(X_test)
weighted_roc_auc_ovo = roc_auc_score(
    y_test, y_prob, multi_class="ovr", average="weighted"
)
weighted_roc_auc_ovo

In [ ]:
acc.append(accuracy_score(y_test, y_pred))
prec.append(precision_score(y_test, y_pred,average='weighted'))
rec.append(recall_score(y_test, y_pred,average='weighted'))
f1.append(f1_score(y_test, y_pred,average='weighted'))
kap.append(cohen_kappa_score(y_test, predictions))
roc.append(weighted_roc_auc_ovo)

# Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(random_state=0,multi_class="multinomial")
clf.fit(X_train,y_train)

In [ ]:
y_pred = clf.predict(X_test)
y_pred 

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.metrics import classification_report
result1 = classification_report(y_test, y_pred)
print("Classification Report:",)
print (result1)
result2 = accuracy_score(y_test,y_pred)
print("Accuracy:",result2)
print(precision_score(y_test, y_pred,average='weighted'))
print(recall_score(y_test, y_pred,average='weighted'))
print(f1_score(y_test, y_pred,average='weighted'))
print(cohen_kappa_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = knn_clf.predict_proba(X_test)
weighted_roc_auc_ovo = roc_auc_score(
    y_test, y_prob, multi_class="ovr", average="weighted"
)
weighted_roc_auc_ovo

In [ ]:
acc.append(accuracy_score(y_test, y_pred))
prec.append(precision_score(y_test, y_pred,average='weighted'))
rec.append(recall_score(y_test, y_pred,average='weighted'))
f1.append(f1_score(y_test, y_pred,average='weighted'))
kap.append(cohen_kappa_score(y_test, predictions))
roc.append(weighted_roc_auc_ovo)

# Naive Bayes

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
p = Pipeline([('Normalizing',MinMaxScaler()),('MultinomialNB',MultinomialNB())])
p.fit(X_train,y_train) 

In [ ]:
y_pred = p.predict(X_test)
y_pred 

In [ ]:
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.metrics import classification_report
result1 = classification_report(y_test, y_pred)
print("Classification Report:",)
print (result1)
result2 = accuracy_score(y_test,y_pred)
print("Accuracy:",result2)
print(precision_score(y_test, y_pred,average='weighted'))
print(recall_score(y_test, y_pred,average='weighted'))
print(f1_score(y_test, y_pred,average='weighted'))
print(cohen_kappa_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import roc_auc_score
y_prob = knn_clf.predict_proba(X_test)
weighted_roc_auc_ovo = roc_auc_score(
    y_test, y_prob, multi_class="ovr", average="weighted"
)
weighted_roc_auc_ovo

In [ ]:
acc.append(accuracy_score(y_test, y_pred))
prec.append(precision_score(y_test, y_pred,average='weighted'))
rec.append(recall_score(y_test, y_pred,average='weighted'))
f1.append(f1_score(y_test, y_pred,average='weighted'))
kap.append(cohen_kappa_score(y_test, predictions))
roc.append(weighted_roc_auc_ovo)

In [ ]:
model_results['Accuracy'] = acc
model_results['Prescision'] = prec
model_results['Recall'] = rec
model_results['F1'] = f1
model_results['Kappa'] = kap
model_results['ROC_AUC'] = roc

In [ ]:
model_results

In [ ]:
fig = model_results.plot(x = "models", y = ['Accuracy',	'Prescision',	'Recall',	'F1',	'Kappa', 'ROC_AUC'], kind = "bar", figsize=(9, 8))
ax = plt.subplot(111)
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_ylim([0.35,1])
plt.show()

# **Results**

---
1. Accuracy
2. Precision
3. Recall
4. F1 Score
5. Cohen Kappa Score
6. Roc auc Score
---
**Random Forest**

0.9573560767590619

0.9584976073991893

0.9573560767590619

0.9552754759579702

---
**Adaboost**

0.802771855010661

0.8534504944314486

0.802771855010661

0.8145437082907355

---
**XGBoost**

0.9658848614072495

0.965915211393339

0.9658848614072495

0.9649037802214644

0.9620538169314109

---
**KNN**

0.9008528784648188

0.9001685697973634

0.9008528784648188

0.8916125571931889

0.8896681342336884

---

**Logistic Regression**

0.9008528784648188

0.900135627354752

0.9008528784648188

0.8995841318871703

0.8897233666267618

---

**Naive Bayes**

0.4275053304904051

0.5715843473005865

0.4275053304904051

0.42702817370427987

0.3624604153007678

---